# GLM-5.3-Flash on Amazon SageMaker AI

Deploy `zai-org/GLM-5.3-Flash` (321 B total / 18 B active, natively FP8) to a SageMaker AI
real-time endpoint, verify it serves, measure it, and tear it down.

Self-contained: it builds the serving container, stages the weights, and creates the endpoint
using nothing but `boto3`.

## Steps

| step | what it does | run time |
| --- | --- | --- |
| 1 | Configure the session | instant |
| 2 | Check the model's memory footprint | seconds |
| 3 | See which one-time setup is already done | seconds |
| 4 | Build and push the serving container | ~10 min |
| 5 | Stage 306 GiB of weights into S3 | ~15 min |
| 6 | Deploy the endpoint | ~15 min |
| 7 | Confirm the engine came up as intended | seconds |
| 8 | Send requests, measure throughput and cost | ~25 min |
| 9 | Delete everything | ~5 min |

Steps 4 and 5 are one-time. Once the image and weights exist, later runs go from step 3
straight to step 6.

## Before you start

You need an AWS account with SageMaker AI access, **quota for an 8-GPU instance type**, an
execution role carrying `AmazonSageMakerFullAccess`, and a Hugging Face token in Secrets
Manager. Install `boto3`; nothing else is required.

Name three resources carefully and the managed IAM policies cover you with no custom policy:

| resource | must be named | why |
| --- | --- | --- |
| ECR repository | `sagemaker-*` | the CodeBuild role permits ECR push only on `repository/sagemaker-*` |
| S3 staging bucket | contains `sagemaker` | `AmazonSageMakerFullAccess` scopes S3 access to `*sagemaker*` |
| Secrets Manager secret | `AmazonSageMaker-*` | that policy scopes `GetSecretValue` to `secret:AmazonSageMaker-*` |

## What this costs

Steps 4 and 5 sit behind `RUN_BUILD` and `RUN_STAGE`, both `False`, so running every cell
cannot start a build or a staging job by accident.

**Step 6 is not gated and provisions an 8-GPU instance.** At the validated instance type that
is $72.795/hr in `us-east-2`. Read step 6 before running it, and do not skip step 9.

---
# Step 1: Configure the session

Edit this cell and nothing else. Everything downstream reads from it.

Account id, bucket name, role ARN and ECR URI are derived at runtime from
`sts:GetCallerIdentity`, so there is nothing account-specific to fill in.

In [ ]:
import json
import time
import boto3
import botocore

PROFILE = None                    # None uses the default credential chain;
                                  # set to a named profile string if you use one
REGION = "us-east-2"

HF_MODEL_ID = "zai-org/GLM-5.3-Flash"
SERVED_MODEL_NAME = "glm-5.3-flash"

ECR_REPO = "sagemaker-vllm"       # must start with 'sagemaker-' (see IAM note above)
IMAGE_TAG = "glm53-flash"

INSTANCE_TYPE = "ml.p5en.48xlarge"

# Ordered fallback, highest priority first. Set to None to pin INSTANCE_TYPE instead.
# Caveat: pools give you ONE endpoint on whichever type had capacity, so they cannot
# produce a three-way hardware comparison.
INSTANCE_POOLS = None

# Engine configuration. These four decide whether the model fits; see step 6.
TENSOR_PARALLEL_SIZE = 4
DATA_PARALLEL_SIZE = 2            # attention replicas. EP = TP x DP
ENABLE_EXPERT_PARALLEL = True     # with TP=4, DP=2 this gives EP=8
MAX_MODEL_LEN = 32768
MAX_NUM_SEQS = 128                # per DP replica, so 256 total here
GPU_MEMORY_UTILIZATION = 0.85
MAX_NUM_BATCHED_TOKENS = 8192

session = boto3.Session(profile_name=PROFILE, region_name=REGION)
sm = session.client("sagemaker")
smr = session.client("sagemaker-runtime")
cw = session.client("cloudwatch")

ACCOUNT = session.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"          # contains 'sagemaker', as required
MODEL_S3_URI = f"s3://{BUCKET}/models/glm-5.3-flash/"
IMAGE_URI = f"{ACCOUNT}.dkr.ecr.{REGION}.amazonaws.com/{ECR_REPO}:{IMAGE_TAG}"
ROLE = (
    f"arn:aws:iam::{ACCOUNT}:role/service-role/"
    "AmazonSageMakerServiceCatalogProductsExecutionRole"
)

print(f"account   {ACCOUNT}")
print(f"region    {REGION}")
print(f"image     {IMAGE_URI}")
print(f"weights   {MODEL_S3_URI}")
print(f"role      {ROLE}")

---
# Step 2: Check the model's memory footprint

Read the real dtype breakdown from the Hub before choosing hardware.

**306 GiB of weights must be resident before the model serves a single token.** That is what
forces an 8-GPU instance, independent of how much throughput you need. No 4-GPU H200 shape
exists on SageMaker or EC2, so 8 GPUs is the floor even for modest traffic.

Architecture facts from `config.json` that shape the serving config:

- 45 layers: 34 KDA linear-attention, 11 DeepSeek-sparse-attention (MLA), 1 MTP layer
- MoE: 288 routed experts, top-8 plus 1 shared, 42 sparse layers
- MLA with `kv_lora_rank` 512 and `qk_rope_head_dim` 0, so NoPE
- Natively multimodal, vision tower included

In [ ]:
import urllib.request

BYTES_PER_DTYPE = {"F8_E4M3": 1, "F8_E5M2": 1, "BF16": 2, "F16": 2, "F32": 4, "I64": 8}

req = urllib.request.Request(f"https://huggingface.co/api/models/{HF_MODEL_ID}")
with urllib.request.urlopen(req, timeout=30) as resp:
    info = json.load(resp)

params = info.get("safetensors", {}).get("parameters", {})
total_bytes = 0
print(f"{'dtype':<10}{'params':>18}{'GiB':>10}")
for dtype, count in sorted(params.items()):
    width = BYTES_PER_DTYPE.get(dtype, 2)
    size = count * width
    total_bytes += size
    print(f"{dtype:<10}{count:>18,}{size / 2**30:>10,.1f}")

total_params = sum(params.values())
print(f"\n{'total':<10}{total_params:>18,}{total_bytes / 2**30:>10,.1f}")
print(f"\nper GPU at 8-way sharding: {total_bytes / 2**30 / 8:,.1f} GiB")

---
# Step 3: See which one-time setup is already done

Steps 4 and 5 run once per account. This checks whether the container image and the staged
weights already exist so you can skip ahead.

If both report present, go straight to step 6.

In [ ]:
# Both stages below are implemented inline in this notebook; nothing external is
# needed. This cell only decides which of them still has work to do.

ecr = session.client("ecr")


def image_exists():
    try:
        ecr.describe_images(
            repositoryName=ECR_REPO, imageIds=[{"imageTag": IMAGE_TAG}]
        )
        return True
    except (ecr.exceptions.RepositoryNotFoundException,
            ecr.exceptions.ImageNotFoundException):
        return False
    except botocore.exceptions.ClientError as exc:
        if exc.response["Error"]["Code"] in ("AccessDeniedException", "AccessDenied"):
            print("  ecr:DescribeImages denied, cannot verify; assuming present")
            return True
        raise


def weights_staged(expected=72):
    prefix = MODEL_S3_URI.replace(f"s3://{BUCKET}/", "")
    resp = session.client("s3").list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return resp.get("KeyCount", 0) >= expected


NEED_BUILD = not image_exists()
NEED_STAGE = not weights_staged()

print(f"container image   {'MISSING, build needed' if NEED_BUILD else 'present'}")
print(f"staged weights    {'MISSING, staging needed' if NEED_STAGE else 'present'}")
if NEED_BUILD or NEED_STAGE:
    print("\nSet RUN_BUILD / RUN_STAGE to True below to run the missing stage(s).")
else:
    print("\nBoth artifacts exist. Steps 4 and 5 will self-skip; go to step 6.")

---
# Step 4: Build and push the serving container

One-time, and no local Docker needed since the build runs in CodeBuild.

Three files define the image. The next cell writes them to `build/`.

The important one is `serve`. It is the entire SageMaker integration: it listens on 8080,
exposes `/ping` and `/invocations` through vLLM's OpenAI server, and translates every
`SM_VLLM_<FLAG>` environment variable into `--<flag>` for `vllm serve`, with `SM_VLLM_MODEL`
becoming the positional argument.

**That is why every engine change in this notebook is an environment edit, never a rebuild.**

In [ ]:
import pathlib

BUILD_DIR = pathlib.Path("build")
BUILD_DIR.mkdir(exist_ok=True)

DOCKERFILE = r"""
# GLM-5.3-Flash (Glm5NextForConditionalGeneration) is not in any tagged vLLM release,
# so the base is the purpose-built dev image.
ARG BASE_IMAGE=vllm/vllm-openai:glm53-flash
FROM ${BASE_IMAGE}

# SageMaker hosting contract: listen on 8080, expose /ping and /invocations.
# vLLM's OpenAI server provides both; serve only translates env vars to CLI flags.
COPY serve /usr/bin/serve
RUN chmod 777 /usr/bin/serve

# HF_HOME must be writable and off the small container layer.
ENV HF_HOME=/tmp/hf \
    TOKENIZERS_PARALLELISM=false

ENTRYPOINT [ "/usr/bin/serve" ]
"""

SERVE = r"""#!/bin/bash
PREFIX="SM_VLLM_"
ARG_PREFIX="--"

# The model is the first positional arg to `vllm serve`; skip it in the loop below.
MODEL_VAR="${PREFIX}MODEL"
MODEL="${!MODEL_VAR}"

# Port 8080 is required by SageMaker.
PORT=(--port 8080)
ARGS=("${PORT[@]}")

while IFS='=' read -r key value; do
    if [ "$key" = "${PREFIX}MODEL" ]; then
        continue
    fi
    # SM_VLLM_MAX_MODEL_LEN -> --max-model-len
    arg_name=$(echo "${key#"${PREFIX}"}" | tr '[:upper:]' '[:lower:]' | tr '_' '-')
    ARGS+=("${ARG_PREFIX}${arg_name}")
    # Bare flags carry no value; "true" means store_true.
    if [ -n "$value" ] && [ "$value" != "true" ] && [ "$value" != "True" ]; then
        ARGS+=("$value")
    fi
done < <(env | grep "^${PREFIX}")

echo "-------------------------------------------------------------------"
echo "vLLM model: [${MODEL}]"
echo "vLLM engine args: [${ARGS[@]}]"
echo "-------------------------------------------------------------------"

if [ -n "$MODEL" ]; then
    exec vllm serve "$MODEL" "${ARGS[@]}"
else
    exec vllm serve "${ARGS[@]}"
fi
"""

BUILDSPEC = r"""
version: 0.2

env:
  variables:
    DOCKER_BUILDKIT: "1"

phases:
  pre_build:
    commands:
      - IMAGE_URI="${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com/${REPOSITORY_NAME}:${IMAGE_TAG}"
      - echo "target=$IMAGE_URI base=$BASE_IMAGE"
      - df -h /
      - aws ecr describe-repositories --repository-names "$REPOSITORY_NAME" --region "$AWS_DEFAULT_REGION"
        || aws ecr create-repository --repository-name "$REPOSITORY_NAME" --region "$AWS_DEFAULT_REGION"
      - aws ecr get-login-password --region "$AWS_DEFAULT_REGION"
        | docker login --username AWS --password-stdin "${AWS_ACCOUNT_ID}.dkr.ecr.${AWS_DEFAULT_REGION}.amazonaws.com"
      - |
        if [ -n "$DOCKERHUB_USER" ] && [ -n "$DOCKERHUB_TOKEN" ]; then
          echo "$DOCKERHUB_TOKEN" | docker login --username "$DOCKERHUB_USER" --password-stdin
        else
          echo "no Docker Hub creds provided; pulling $BASE_IMAGE anonymously"
        fi

  build:
    commands:
      # --platform is explicit: the base tag is a multi-arch manifest and the target is x86.
      - docker build --platform linux/amd64 --build-arg "BASE_IMAGE=${BASE_IMAGE}" --file Dockerfile --tag "$IMAGE_URI" .
      # Fail here, loudly, rather than at endpoint creation if serve did not land.
      - docker run --rm --entrypoint /bin/sh "$IMAGE_URI" -c "test -x /usr/bin/serve && head -1 /usr/bin/serve"
      - docker run --rm --entrypoint /bin/sh "$IMAGE_URI" -c "python3 -c 'import vllm; print(\"vllm\", vllm.__version__)'"

  post_build:
    commands:
      - docker push "$IMAGE_URI"
      # Deliberately non-fatal: the SageMaker CodeBuild service role is not granted
      # ecr:DescribeImages, so this returns 254 even on a fully successful push.
      - aws ecr describe-images --repository-name "$REPOSITORY_NAME" --image-ids "imageTag=$IMAGE_TAG" --region "$AWS_DEFAULT_REGION" || echo "(describe-images not permitted; the push itself succeeded)"
      - echo "pushed $IMAGE_URI"
"""

(BUILD_DIR / "Dockerfile").write_text(DOCKERFILE.lstrip(), encoding="utf-8")
(BUILD_DIR / "serve").write_text(SERVE, encoding="utf-8", newline="\n")
(BUILD_DIR / "buildspec.yml").write_text(BUILDSPEC.lstrip(), encoding="utf-8")

for path in sorted(BUILD_DIR.iterdir()):
    print(f"  {path}  {path.stat().st_size:>6,} bytes")
print("\nserve is written with LF line endings, which a shell script requires.")

### Build it

The notebook zips the three files to S3, creates a CodeBuild project with `privilegedMode`
(required to run a Docker daemon), starts the build and polls it.

`BUILD_GENERAL1_LARGE` is deliberate: the compressed base image is about 8.6 GB and smaller
compute types run out of disk. Expect roughly 10 minutes.

Set `RUN_BUILD = True` to actually build.

In [ ]:
import io
import zipfile

RUN_BUILD = False    # flip to True to actually build; a build bills CodeBuild minutes

CODEBUILD_PROJECT = "glm53-flash-image"
BASE_IMAGE = "vllm/vllm-openai:glm53-flash"
CODEBUILD_ROLE = (
    f"arn:aws:iam::{ACCOUNT}:role/service-role/"
    "AmazonSageMakerServiceCatalogProductsCodeBuildRole"
)

cb = session.client("codebuild")


def upload_source_bundle():
    """Zip Dockerfile + serve + buildspec.yml and put it in S3 as the build source."""
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for name in ("Dockerfile", "serve", "buildspec.yml"):
            archive.write(BUILD_DIR / name, arcname=name)
    key = f"codebuild/glm53-flash/{int(time.time())}/source.zip"
    session.client("s3").put_object(Bucket=BUCKET, Key=key, Body=buffer.getvalue())
    print(f"source bundle  s3://{BUCKET}/{key}  ({buffer.tell():,} bytes)")
    return key


def ensure_project(source_key):
    spec = {
        "name": CODEBUILD_PROJECT,
        "source": {
            "type": "S3",
            "location": f"{BUCKET}/{source_key}",
            "buildspec": "buildspec.yml",
        },
        "artifacts": {"type": "NO_ARTIFACTS"},
        "environment": {
            "type": "LINUX_CONTAINER",
            "image": "aws/codebuild/standard:7.0",
            "computeType": "BUILD_GENERAL1_LARGE",   # disk headroom for the 8.6 GB base
            "privilegedMode": True,                  # required: we run a Docker daemon
            "environmentVariables": [
                {"name": "AWS_ACCOUNT_ID", "value": ACCOUNT},
                {"name": "REPOSITORY_NAME", "value": ECR_REPO},
                {"name": "IMAGE_TAG", "value": IMAGE_TAG},
                {"name": "BASE_IMAGE", "value": BASE_IMAGE},
            ],
        },
        "serviceRole": CODEBUILD_ROLE,
        "timeoutInMinutes": 60,
    }
    try:
        cb.create_project(**spec)
        print(f"created CodeBuild project {CODEBUILD_PROJECT}")
    except cb.exceptions.ResourceAlreadyExistsException:
        cb.update_project(**spec)
        print(f"updated CodeBuild project {CODEBUILD_PROJECT}")


def run_build():
    build_id = cb.start_build(projectName=CODEBUILD_PROJECT)["build"]["id"]
    print(f"build started   {build_id}")
    started, last = time.time(), None
    while True:
        build = cb.batch_get_builds(ids=[build_id])["builds"][0]
        status, phase = build["buildStatus"], build.get("currentPhase")
        if (status, phase) != last:
            print(f"  [{time.time() - started:>5.0f}s] {status} {phase}")
            last = (status, phase)
        if status != "IN_PROGRESS":
            return status
        time.sleep(20)


if not RUN_BUILD:
    print(f"RUN_BUILD is False. Image needed: {NEED_BUILD}")
elif not NEED_BUILD:
    print("image already present, skipping build")
else:
    ensure_project(upload_source_bundle())
    print(f"\nfinal status: {run_build()}")
    print(f"image: {IMAGE_URI}")

---
# Step 5: Stage the weights into S3

One-time, and **required rather than an optimisation.**

This architecture's multimodal processor reads `processor_config.json` from a local directory,
so `SM_VLLM_MODEL` must point at a path on disk (`/opt/ml/model`) rather than a Hub repo id.
Staging also splits the work across two separate 3600 s budgets,
`ModelDataDownloadTimeoutInSeconds` for the copy and
`ContainerStartupHealthCheckTimeoutInSeconds` for engine init, instead of racing 306 GiB of
download and engine startup inside one.

The worker below streams each file from the Hub straight into S3 multipart, never landing a
file on disk, which is how a 30 GB volume moves 306 GiB. Four details matter:

- **Fetch `resolve/<rev>/<file>` with an `Authorization` header.** This bypasses Xet, whose
  unauthenticated path throttles heavily.
- **Set `AWS_DEFAULT_REGION` in the job environment.** Processing containers have no implicit
  region and boto3 raises `NoRegionError` without it.
- **Pass only the secret *id*.** The worker resolves the token at runtime, so
  `DescribeProcessingJob` never exposes it.
- **Use `RLock` for the shared counter**, so a worker cannot deadlock against itself.

Measured: **305.8 GiB in 14.8 minutes at 353 MiB/s** with 16 workers.

Set `RUN_STAGE = True` to launch it.

In [ ]:
STAGER = r"""
import json
import os
import threading
import urllib.request
from concurrent.futures import ThreadPoolExecutor

import boto3

REPO = os.environ["HF_REPO"]
DEST = os.environ["S3_DEST"]
WORKERS = int(os.environ.get("MAX_WORKERS", "8"))
SECRET_ID = os.environ.get("HF_TOKEN_SECRET_ID") or ""

session = boto3.Session()
s3 = session.client("s3")

token = ""
if SECRET_ID:
    raw = session.client("secretsmanager").get_secret_value(
        SecretId=SECRET_ID
    )["SecretString"]
    try:
        token = json.loads(raw).get("token", raw)
    except json.JSONDecodeError:
        token = raw
    os.environ["HF_TOKEN"] = token

bucket, prefix = DEST[len("s3://"):].split("/", 1)

request = urllib.request.Request(f"https://huggingface.co/api/models/{REPO}")
if token:
    request.add_header("Authorization", "Bearer " + token)
with urllib.request.urlopen(request, timeout=60) as response:
    files = [s["rfilename"] for s in json.load(response)["siblings"]]
print("%d files to move" % len(files), flush=True)

# RLock so a worker can take the counter lock re-entrantly without self-deadlock.
counter_lock = threading.RLock()
state = {"done": 0, "bytes": 0}


def move(name):
    url = "https://huggingface.co/%s/resolve/main/%s" % (REPO, name)
    req = urllib.request.Request(url)
    if token:
        req.add_header("Authorization", "Bearer " + token)
    # Stream into S3 multipart. boto3 handles the part splitting for a
    # non-seekable stream, so nothing is written to the local volume.
    with urllib.request.urlopen(req, timeout=1800) as body:
        size = int(body.headers.get("Content-Length") or 0)
        s3.upload_fileobj(body, bucket, prefix + name)
    with counter_lock:
        state["done"] += 1
        state["bytes"] += size
        print("  [%d/%d] %s (%.1f GiB total)"
              % (state["done"], len(files), name, state["bytes"] / 2**30),
              flush=True)


errors = []
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for name, result in zip(files, pool.map(move, files, timeout=None)):
        pass

print("staged %d/%d files, %.1f GiB, to %s"
      % (state["done"], len(files), state["bytes"] / 2**30, DEST), flush=True)
if state["done"] != len(files):
    raise SystemExit("incomplete: %d of %d" % (state["done"], len(files)))
"""

(BUILD_DIR / "stage_weights.py").write_text(STAGER.lstrip(), encoding="utf-8")
print(f"wrote {BUILD_DIR / 'stage_weights.py'}  "
      f"{(BUILD_DIR / 'stage_weights.py').stat().st_size:,} bytes")

import ast
ast.parse((BUILD_DIR / "stage_weights.py").read_text(encoding="utf-8"))
print("worker script parses")

In [ ]:
RUN_STAGE = False   # flip to True to launch; this bills an ml.m5.12xlarge Processing job

# Secrets Manager id holding {"token": "hf_..."}. Only the id travels in the job
# definition; the worker resolves the token at runtime. Must be named
# AmazonSageMaker-* or AmazonSageMakerFullAccess will not permit GetSecretValue.
HF_TOKEN_SECRET = "AmazonSageMaker-hf-token"
STAGE_INSTANCE = "ml.m5.12xlarge"
STAGE_WORKERS = 16
STAGE_VOLUME_GB = 30       # the job streams; it never lands the files


def launch_staging():
    job_name = f"glm53-flash-stage-{int(time.time())}"
    code_key = f"processing/glm53-flash/{job_name}/stage_weights.py"

    session.client("s3").put_object(
        Bucket=BUCKET,
        Key=code_key,
        Body=(BUILD_DIR / "stage_weights.py").read_bytes(),
    )

    sm.create_processing_job(
        ProcessingJobName=job_name,
        RoleArn=ROLE,
        AppSpecification={
            "ImageUri": IMAGE_URI,
            # Overrides the image's /usr/bin/serve entrypoint.
            "ContainerEntrypoint": [
                "python3", "/opt/ml/processing/input/code/stage_weights.py"
            ],
        },
        ProcessingInputs=[{
            "InputName": "code",
            "S3Input": {
                "S3Uri": f"s3://{BUCKET}/{code_key}",
                "LocalPath": "/opt/ml/processing/input/code",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
                "S3DataDistributionType": "FullyReplicated",
            },
        }],
        ProcessingResources={"ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": STAGE_INSTANCE,
            "VolumeSizeInGB": STAGE_VOLUME_GB,
        }},
        StoppingCondition={"MaxRuntimeInSeconds": 6 * 3600},
        Environment={
            "HF_REPO": HF_MODEL_ID,
            "S3_DEST": MODEL_S3_URI,
            "MAX_WORKERS": str(STAGE_WORKERS),
            # Processing containers have no implicit region.
            "AWS_DEFAULT_REGION": REGION,
            # Only the secret id travels in the job definition, never the token.
            "HF_TOKEN_SECRET_ID": HF_TOKEN_SECRET,
        },
    )
    print(f"processing job started  {job_name}")
    print(f"  logs: /aws/sagemaker/ProcessingJobs -> {job_name}")

    started, last = time.time(), None
    while True:
        desc = sm.describe_processing_job(ProcessingJobName=job_name)
        status = desc["ProcessingJobStatus"]
        if status != last:
            print(f"  [{time.time() - started:>5.0f}s] {status}")
            last = status
        if status in ("Completed", "Failed", "Stopped"):
            if status != "Completed":
                print(f"  reason: {desc.get('FailureReason')}")
            return status
        time.sleep(30)


if not RUN_STAGE:
    print(f"RUN_STAGE is False. Weights needed: {NEED_STAGE}")
elif not NEED_STAGE:
    print("weights already staged, skipping")
else:
    print(f"final status: {launch_staging()}")
    print("measured on the real run: 305.8 GiB in 14.8 min at 353 MiB/s")

In [ ]:
s3 = session.client("s3")
prefix = MODEL_S3_URI.replace(f"s3://{BUCKET}/", "")

objects, token, total = [], None, 0
while True:
    kwargs = {"Bucket": BUCKET, "Prefix": prefix}
    if token:
        kwargs["ContinuationToken"] = token
    page = s3.list_objects_v2(**kwargs)
    for obj in page.get("Contents", []):
        objects.append(obj["Key"].rsplit("/", 1)[-1])
        total += obj["Size"]
    token = page.get("NextContinuationToken")
    if not token:
        break

print(f"objects        {len(objects)}")
print(f"total size     {total / 2**30:,.1f} GiB")

# processor_config.json is required by this architecture and is easy to miss.
required = ["processor_config.json", "config.json", "tokenizer_config.json"]
for name in required:
    mark = "present" if name in objects else "MISSING"
    print(f"{name:<28}{mark}")

shards = [o for o in objects if o.endswith(".safetensors")]
print(f"safetensors shards           {len(shards)}")

---
# Step 6: Deploy the endpoint

**This step provisions an 8-GPU instance and starts billing.**

### Choosing the topology

vLLM derives expert-parallel width rather than taking it directly: **EP = TP x DP**. So
"TP=4, EP=2" is not expressible; on 8 GPUs the EP config is TP=4, DP=2 with expert parallel on,
giving EP=8. Note `--max-num-seqs` is **per DP replica**, so 128 here means 256 total.

Per-GPU weight cost barely moves between topologies, so choose on throughput, not memory:

| topology | EP | experts/GPU | other/GPU | total/GPU | attention replicas |
| --- | --- | --- | --- | --- | --- |
| TP=8, DP=1 | off | 36.6 GiB | 1.7 GiB | 38.3 GiB | 1 |
| TP=4, DP=2, EP | 8 | 36.6 GiB | 3.3 GiB | 39.9 GiB | 2 |
| TP=2, DP=4, EP | 8 | 36.6 GiB | 6.6 GiB | 43.2 GiB | 4 |

**DP=2 is not two replicas.** The 288 routed experts shard exactly once across all 8 GPUs;
only attention and dense weights duplicate per DP group. One endpoint, one instance, one copy
of the expert weights, so setting DP=1 does not reduce cost.

### Why DP is the lever on KV capacity

MLA caches a compressed latent that is not head-partitioned, so vLLM **replicates it on every
tensor-parallel rank**. Measured at TP=8, `82.07 GiB / 7,253,058 tokens = 11.86 KiB per token`,
and that pool is one GPU's KV memory rather than eight.

- Raising **TP does not raise KV capacity**, it duplicates the same cache more times.
- Raising **DP does**, because each attention replica holds a distinct cache.

Confirmed twice: DP=2 raised total usable KV from 7,253,058 to **9,857,994 tokens** even with
utilisation reduced from 0.90 to 0.85.

To check this on any new config, divide `Available KV cache memory` by `GPU KV cache size` and
see whether the pool tracks one GPU or all of them.

In [ ]:
CONTAINER_MODEL_DIR = "/opt/ml/model"
MAX_DOWNLOAD_TIMEOUT = 3600      # service cap
MAX_STARTUP_TIMEOUT = 3600       # service cap


def build_environment():
    """SM_VLLM_* becomes `vllm serve --<flag>`; everything else is plain container env."""
    env = {
        # positional model argument. A local directory, never a repo id (see step 5)
        "SM_VLLM_MODEL": CONTAINER_MODEL_DIR,
        "SM_VLLM_SERVED_MODEL_NAME": SERVED_MODEL_NAME,
        # sharding
        "SM_VLLM_TENSOR_PARALLEL_SIZE": str(TENSOR_PARALLEL_SIZE),
        # the two knobs that decide whether this fits
        "SM_VLLM_MAX_MODEL_LEN": str(MAX_MODEL_LEN),
        "SM_VLLM_MAX_NUM_SEQS": str(MAX_NUM_SEQS),
        # memory and scheduling
        "SM_VLLM_GPU_MEMORY_UTILIZATION": str(GPU_MEMORY_UTILIZATION),
        "SM_VLLM_MAX_NUM_BATCHED_TOKENS": str(MAX_NUM_BATCHED_TOKENS),
        # bound the vision path so profiling does not reserve for the worst case
        "SM_VLLM_LIMIT_MM_PER_PROMPT": json.dumps({"image": 4, "video": 0}),
        # Keep the KV cache at the model dtype. This architecture is NoPE MLA
        # (qk_rope_head_dim=0), which the FP8 MLA cache kernel does not support, so
        # an fp8 value here is not a valid option for this model.
        "SM_VLLM_KV_CACHE_DTYPE": "auto",
        # plain container env, no SM_VLLM_ prefix so not passed to vllm
        "HF_HOME": "/tmp/hf",
        "TOKENIZERS_PARALLELISM": "false",
        # Not needed when weights come from S3, but set as a matter of course: if
        # anything resolves from the Hub the default filters fetch weights only, and
        # this architecture also needs processor_config.json and the chat template.
        "HF_HUB_DOWNLOAD_ALLOW_PATTERNS": "*.json,*.jinja,*.safetensors,*.model,*.txt,*.py",
    }
    if DATA_PARALLEL_SIZE > 1:
        env["SM_VLLM_DATA_PARALLEL_SIZE"] = str(DATA_PARALLEL_SIZE)
    if ENABLE_EXPERT_PARALLEL:
        env["SM_VLLM_ENABLE_EXPERT_PARALLEL"] = "true"
    return env


def render_vllm_command(env):
    """Reproduce what `serve` will exec. Review this before spending an hour of GPU time."""
    parts = ["vllm serve", env["SM_VLLM_MODEL"], "--port 8080"]
    for key, value in env.items():
        if not key.startswith("SM_VLLM_") or key == "SM_VLLM_MODEL":
            continue
        flag = "--" + key[len("SM_VLLM_"):].lower().replace("_", "-")
        if value in ("true", "True"):
            parts.append(flag)                    # store_true flags take no value
        elif any(ch in value for ch in ' {}"'):
            parts.append(f"{flag} '{value}'")     # e.g. --limit-mm-per-prompt JSON
        else:
            parts.append(f"{flag} {value}")
    return " \\\n    ".join(parts)


ENV = build_environment()
print(render_vllm_command(ENV))

gib_per_gpu = 306 / 8
print(f"\nweights per GPU     {gib_per_gpu:,.1f} GiB")
print(f"EP width            {TENSOR_PARALLEL_SIZE * DATA_PARALLEL_SIZE if ENABLE_EXPERT_PARALLEL else 'off'}")
print(f"total max_num_seqs  {MAX_NUM_SEQS * DATA_PARALLEL_SIZE}")

### Create the model, endpoint config and endpoint

`CompressionType: "None"` matters: the weights are already uncompressed shards, so SageMaker
mounts the prefix at `/opt/ml/model` rather than trying to expand an archive.

`RoutingConfig` uses `LEAST_OUTSTANDING_REQUESTS` because LLM requests have very uneven
service times and random routing sends work to already-busy replicas.

In [ ]:
suffix = f"{int(time.time())}"
label = INSTANCE_TYPE.replace("ml.", "").replace(".", "-")
MODEL_NAME = f"glm53flash-{label}-{suffix}"
CONFIG_NAME = f"glm53flash-{label}-epc-{suffix}"
ENDPOINT_NAME = f"glm53flash-{label}-ep-{suffix}"

sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=ROLE,
    PrimaryContainer={
        "Image": IMAGE_URI,
        "Environment": ENV,
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": MODEL_S3_URI,
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
    },
)
print(f"model created            {MODEL_NAME}")

variant = {
    "VariantName": "AllTraffic",
    "ModelName": MODEL_NAME,
    "InitialInstanceCount": 1,
    "InitialVariantWeight": 1.0,
    "ModelDataDownloadTimeoutInSeconds": MAX_DOWNLOAD_TIMEOUT,
    "ContainerStartupHealthCheckTimeoutInSeconds": MAX_STARTUP_TIMEOUT,
    "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
    # No VolumeSizeInGB: these instance types ship local NVMe, and SageMaker rejects
    # the parameter for instance types that provide their own instance storage.
}

if INSTANCE_POOLS:
    # Ordered fallback, up to 5 types. Replaces InstanceType; setting both is invalid.
    variant["InstancePools"] = [
        {"InstanceType": t, "Priority": i} for i, t in enumerate(INSTANCE_POOLS)
    ]
    variant["VariantInstanceProvisionTimeoutInSeconds"] = 3600   # 300-3600
else:
    variant["InstanceType"] = INSTANCE_TYPE

sm.create_endpoint_config(EndpointConfigName=CONFIG_NAME, ProductionVariants=[variant])
print(f"endpoint config created  {CONFIG_NAME}")

sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=CONFIG_NAME)
print(f"endpoint creating        {ENDPOINT_NAME}")
print(f"logs                     /aws/sagemaker/Endpoints/{ENDPOINT_NAME}")

### Wait for it to come up

Expect roughly 15 minutes. If no instance is available, `InsufficientInstanceCapacity` takes
about 31 minutes to surface, and note that `VariantInstanceProvisionTimeoutInSeconds` applies
to instance pools rather than to a pinned single instance type.

The loop retries transient connection errors and leaves the endpoint in place either way, so
`DescribeEndpoint` and CloudWatch stay available to inspect.

In [ ]:
def wait_for_endpoint(name, timeout=3900, poll=30):
    """Poll until the endpoint reaches a terminal state and return its description."""
    started = time.time()
    transient = 0
    while time.time() - started < timeout:
        try:
            desc = sm.describe_endpoint(EndpointName=name)
            transient = 0
        except (botocore.exceptions.EndpointConnectionError,
                botocore.exceptions.ConnectionClosedError,
                botocore.exceptions.ReadTimeoutError) as exc:
            # A local connectivity blip says nothing about the deployment, so keep
            # polling rather than treating it as a result.
            transient += 1
            print(f"  [{time.time() - started:>5.0f}s] transient {type(exc).__name__} "
                  f"(#{transient}), retrying")
            time.sleep(poll)
            continue

        status = desc["EndpointStatus"]
        print(f"  [{time.time() - started:>5.0f}s] {status}")
        if status in ("InService", "Failed", "OutOfService"):
            if status != "InService":
                print(f"\nFailureReason: {desc.get('FailureReason', '(none reported)')}")
            return desc
        time.sleep(poll)

    print("\npoll timeout. The endpoint is left in place so DescribeEndpoint still")
    print("reports its status and FailureReason.")
    return sm.describe_endpoint(EndpointName=name)


desc = wait_for_endpoint(ENDPOINT_NAME)

if desc["EndpointStatus"] == "InService":
    variants = desc["ProductionVariants"][0]
    # With InstancePools, this is how you learn which type actually got capacity.
    print(f"\nserving on {variants.get('InstancePools') or INSTANCE_TYPE}")

---
# Step 7: Confirm the engine came up as intended

Read what the engine did rather than trusting the config. vLLM's worker names encode the
topology, so TP=4/DP=2/EP=8 gives `Worker_DP0_TP0_EP0` through `Worker_DP1_TP3_EP7`.

Measured on that configuration:

```
Available KV cache memory: 71.31 GiB            (per GPU)
GPU KV cache size: 4,928,997 tokens             (per DP replica, x2 = 9,857,994)
Maximum concurrency for 32,768 tokens per request: 150.42x
init engine (profile, create kv cache, warmup model) took 404.4 s
```

In [ ]:
logs = session.client("logs")
group = f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}"

PATTERNS = [
    "Loading weights took",
    "Available KV cache memory",
    "GPU KV cache size",
    "Maximum concurrency",
    "init engine",
    "Application startup complete",
]

for pattern in PATTERNS:
    try:
        events = logs.filter_log_events(
            logGroupName=group, filterPattern=f'"{pattern}"', limit=3
        )["events"]
    except logs.exceptions.ResourceNotFoundException:
        print(f"log group not found yet: {group}")
        break
    for event in events:
        print(event["message"].strip())

# The check that generalises: does the KV pool track one GPU or all of them?
print("\nDivide 'Available KV cache memory' by 'GPU KV cache size' for KiB/token.")
print("Measured 11.86 KiB/token at TP=8, matching the config.json estimate to 4%.")

---
# Step 8: Send requests and measure

The container speaks the OpenAI chat completions API, so an `/invocations` payload is an
OpenAI request body.

**This model defaults to `reasoning_effort=max`,** so responses open with reasoning tokens.
Any throughput figure counts them, and TTFT is time to the first *reasoning* token.
`reasoning_effort` is a per-request field, so `"low"`, `"high"` or `"max"` can be set per call.
Measured separately, `max` emits about **1.46x** the output tokens of `low` on identical
prompts.

In [ ]:
payload = {
    "model": SERVED_MODEL_NAME,
    "messages": [{"role": "user", "content": "In one sentence, what is MLA attention?"}],
    "max_tokens": 128,
    "temperature": 0.0,
}

started = time.perf_counter()
resp = smr.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload),
)
elapsed = time.perf_counter() - started
body = json.loads(resp["Body"].read())

print(f"round trip     {elapsed * 1000:,.0f} ms")
print(f"usage          {body.get('usage')}")
print(f"\n{body['choices'][0]['message']['content']}")

In [ ]:
# Streaming, which is how you get a real TTFT. Measured 232 ms on the p5en bring-up.
payload_stream = dict(payload, stream=True)

started = time.perf_counter()
ttft = None
chunks = 0
text = []

resp = smr.invoke_endpoint_with_response_stream(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps(payload_stream),
)

for event in resp["Body"]:
    if "PayloadPart" not in event:
        continue
    for line in event["PayloadPart"]["Bytes"].decode("utf-8").splitlines():
        if not line.startswith("data: "):
            continue
        data = line[len("data: "):].strip()
        if data == "[DONE]":
            continue
        delta = json.loads(data)["choices"][0].get("delta", {})
        piece = delta.get("content") or delta.get("reasoning_content") or ""
        if piece:
            if ttft is None:
                ttft = time.perf_counter() - started
            chunks += 1
            text.append(piece)

total = time.perf_counter() - started
print(f"TTFT           {ttft * 1000:,.0f} ms" if ttft else "no tokens received")
print(f"chunks         {chunks}")
print(f"total          {total:,.1f} s")
print(f"\n{''.join(text)[:600]}")

### Run a concurrency sweep

Both paths use **Amazon SageMaker AI optimized generative AI benchmarking**. The difference is
how you drive it: the Python SDK wrapper, or the underlying `CreateAIBenchmarkJob` API. Prefer
the raw API, because `concurrency` accepts a **list** so one job sweeps every level (21 min
total against roughly 17 min *per point*), results persist in S3 independently of the job, and
it returns end-to-end latency percentiles the wrapper omits.

Four practical rules: write each point as it completes, bound the teardown, send warm-up
requests so the first point is not measuring cold start, and keep the jobs so percentiles stay
retrievable.

### Measured results

`ml.p5en.48xlarge`, TP=4/DP=2/EP=8, 8192 in / 256 out, streaming:

| conc | TTFT p50 | TTFT p90 | ITL p50 | out tok/s | req/s |
| --- | --- | --- | --- | --- | --- |
| 16 | 1428 ms | 12,316 ms | 20.1 ms | 311 | 1.2 |
| 32 | 979 ms | 4,672 ms | 25.0 ms | 842 | 3.3 |
| 64 | 634 ms | 2,724 ms | 32.3 ms | 1,458 | 5.7 |
| 128 | **415 ms** | 1,561 ms | 43.4 ms | **2,359** | 9.3 |

Two things to read carefully. **The tail is 9x the median at low concurrency.** With
8192-token inputs and `--max-num-batched-tokens 8192` only one prefill fits per scheduler step,
so a queued request waits whole prefill cycles. If your traffic sits at low concurrency,
raising `--max-num-batched-tokens` is the lever to test.

And **output was capped at 256 tokens** while a `reasoning_effort=max` response runs to a p50
of about 2053, so these responses were truncated inside the reasoning trace. The throughput
figures describe reasoning-token production. ITL is unaffected.

### GPU utilisation

SageMaker publishes GPU telemetry to CloudWatch for any endpoint, so you can recover it for
any window, including a benchmark that has already finished.

**`GPUUtilization` is summed across GPUs.** The ceiling on an 8-GPU instance is 800%, not
100%, and `CPUUtilization` is likewise summed across vCPUs. Divide, or the numbers look
impossible.

Over the sweep window above, per-GPU utilisation sat at **94 to 99% from the first minute of
load, at every concurrency level including 16**. The rise from 311 to 2,359 tok/s therefore
came from better batching efficiency on already-busy GPUs, not from filling idle capacity.
That also means a "percentage of peak throughput" describes batch occupancy rather than
reclaimable hardware.

`GPUMemoryUtilization` stays flat near 96.8% because vLLM pre-allocates the KV pool at
startup, so it reflects `--gpu-memory-utilization`, not load.

In [ ]:
from datetime import datetime, timedelta, timezone

GPUS = 8
end = datetime.now(timezone.utc)
start = end - timedelta(minutes=30)

for metric in ("GPUUtilization", "GPUMemoryUtilization"):
    points = cw.get_metric_statistics(
        Namespace="/aws/sagemaker/Endpoints",
        MetricName=metric,
        Dimensions=[
            {"Name": "EndpointName", "Value": ENDPOINT_NAME},
            {"Name": "VariantName", "Value": "AllTraffic"},
        ],
        StartTime=start,
        EndTime=end,
        Period=60,
        Statistics=["Average", "Maximum"],
    )["Datapoints"]

    if not points:
        print(f"{metric}: no datapoints yet")
        continue

    avg = sum(p["Average"] for p in points) / len(points)
    peak = max(p["Maximum"] for p in points)
    print(f"{metric:<24} raw avg {avg:>7.1f}%   per-GPU avg {avg / GPUS:>5.1f}%"
          f"   per-GPU max {peak / GPUS:>5.1f}%")

### Cost per token

One Pricing API detail: SageMaker uses the **`instanceName`** attribute, not `instanceType`.
Filtering on `instanceType` returns zero results with no error, which looks exactly like "no
price exists". The Pricing API also lives in `us-east-1` regardless of the region you price.

In [ ]:
pricing = session.client("pricing", region_name="us-east-1")


def hourly_rate(instance_type, location="US East (Ohio)"):
    """Real-time endpoint hosting rate. Note: instanceName, NOT instanceType."""
    pages = pricing.get_paginator("get_products").paginate(
        ServiceCode="AmazonSageMaker",
        Filters=[
            {"Type": "TERM_MATCH", "Field": "instanceName", "Value": instance_type},
            {"Type": "TERM_MATCH", "Field": "location", "Value": location},
        ],
    )
    for page in pages:
        for raw in page["PriceList"]:
            item = json.loads(raw)
            usage = item["product"]["attributes"].get("usagetype", "")
            if "Host:" not in usage:            # Host = real-time endpoint
                continue
            for term in item["terms"]["OnDemand"].values():
                for dim in term["priceDimensions"].values():
                    return float(dim["pricePerUnit"]["USD"]), usage
    return None, None


rate, usage = hourly_rate(INSTANCE_TYPE)
print(f"{INSTANCE_TYPE}  ${rate:,.4f}/hr  ({usage})\n")

# Cost at each measured concurrency level. Output-only is the number a customer feels;
# total includes prompt tokens, which are cheap to process but real.
MEASURED = {16: 311, 32: 842, 64: 1458, 128: 2359}   # concurrency -> output tok/s
INPUT_TOKENS, OUTPUT_TOKENS = 8192, 256

print(f"{'conc':>6}{'out tok/s':>12}{'$/1M out':>12}{'$/1M total':>13}")
for conc, tps in MEASURED.items():
    per_hour = tps * 3600
    dollars_out = rate / (per_hour / 1e6)
    total_tps = tps * (INPUT_TOKENS + OUTPUT_TOKENS) / OUTPUT_TOKENS
    dollars_total = rate / (total_tps * 3600 / 1e6)
    print(f"{conc:>6}{tps:>12,}{dollars_out:>12,.2f}{dollars_total:>13,.3f}")

print("\nThese assume a continuously loaded endpoint, which the section 6 telemetry")
print("supports while under load. Over an endpoint's whole life, duty cycle governs")
print("real cost per token far more than any engine setting does.")

---
# Step 9: Delete everything

**An idle endpoint bills at the full hourly rate**, $72.795/hr at the validated instance type,
roughly $1,750 a day. Delete in order: endpoint, then config, then model.

In [ ]:
for delete, name, kwarg in (
    (sm.delete_endpoint, ENDPOINT_NAME, "EndpointName"),
    (sm.delete_endpoint_config, CONFIG_NAME, "EndpointConfigName"),
    (sm.delete_model, MODEL_NAME, "ModelName"),
):
    try:
        delete(**{kwarg: name})
        print(f"deleted {name}")
    except botocore.exceptions.ClientError as exc:
        print(f"skip {name}: {exc.response['Error']['Message']}")